In [1]:
# Jokioinen Ilmala (FMISID=101104) — Daily weather 1991–2022
# Parse FMI WFS *simple feature* response (BsWfs) and pivot to rrday, snow, tday, tmin, tmax

import requests, time
from xml.etree import ElementTree as ET
import pandas as pd

FMISID  = 101104
START_Y = 1991
END_Y   = 2022
OUT_CSV = "jokioinen_daily_1991_2022.csv"
OUT_XLS = "jokioinen_daily_1991_2022.xlsx"

BASE_URL = "https://opendata.fmi.fi/wfs"     # no key needed
STORED_ID = "fmi::observations::weather::daily::simple"
UA = {"User-Agent": "Jupyter-FMI-WFS/1.0 (+https://opendata.fmi.fi)"}

# namespaces
NS = {
    "wfs":  "http://www.opengis.net/wfs/2.0",
    "gml":  "http://www.opengis.net/gml/3.2",
    "BsWfs":"http://xml.fmi.fi/schema/wfs/2.0",
    "xsi":  "http://www.w3.org/2001/XMLSchema-instance",
}

# we will keep only these variables; others (e.g. TG_PT12H_min) are ignored
TARGET_VARS = ["rrday", "snow", "tday", "tmin", "tmax"]

def fetch_daily_simple(year: int) -> pd.DataFrame:
    start_iso = f"{year:04d}-01-01T00:00:00Z"
    end_iso   = f"{(year+1 if year < END_Y else 2023):04d}-01-01T00:00:00Z"

    params = {
        "service": "WFS",
        "version": "2.0.0",
        "request": "getFeature",
        "storedquery_id": STORED_ID,
        "fmisid": str(FMISID),
        "starttime": start_iso,
        "endtime": end_iso,
        "timestep": "1440",     # 1 day alignment
    }

    r = requests.get(BASE_URL, params=params, headers=UA, timeout=120)
    r.raise_for_status()

    # Parse the *simple* schema: BsWfs:BsWfsElement items
    root = ET.fromstring(r.content)

    # If server sent an OWS ExceptionReport (unlikely with daily::simple), surface it
    if root.tag.endswith("ExceptionReport"):
        texts = [el.text for el in root.findall(".//{*}ExceptionText")]
        raise RuntimeError("OWS ExceptionReport: " + " | ".join([t for t in texts if t]))

    rows = []  # will collect (date, name, value)
    for member in root.findall(".//wfs:member", NS):
        el = member.find("./BsWfs:BsWfsElement", NS)
        if el is None:
            # some deployments use element names like BsWfs:BsWfsObservation; fallback: find any child in BsWfs ns
            el = next((child for child in member if child.tag.startswith("{"+NS["BsWfs"]+"}")), None)
        if el is None:
            continue

        time_el = el.find("./BsWfs:Time", NS)
        name_el = el.find("./BsWfs:ParameterName", NS)
        val_el  = el.find("./BsWfs:ParameterValue", NS)
        if time_el is None or name_el is None:
            continue

        d = pd.to_datetime(time_el.text, utc=True, errors="coerce")
        if pd.isna(d):
            continue
        day = d.date()

        name = (name_el.text or "").strip().lower()
        # Some deployments may return uppercase/labeled names; normalize few aliases
        alias = {"precipitationamount": "rrday", "precipitation": "rrday", "snowdepth": "snow",
                 "tavg": "tday", "tmean": "tday"}
        name = alias.get(name, name)

        val = None
        if val_el is not None and val_el.text and val_el.text.strip() != "":
            try:
                val = float(val_el.text.strip())
            except ValueError:
                val = None

        rows.append((day, name, val))

    if not rows:
        # Empty year: return a properly shaped empty frame
        return pd.DataFrame(columns=["date"] + TARGET_VARS).set_index("date")

    # Build long -> wide
    long_df = pd.DataFrame(rows, columns=["date", "variable", "value"])
    # keep only target variables; you can add more here if needed
    long_df = long_df[long_df["variable"].isin(TARGET_VARS)]
    wide = long_df.pivot_table(index="date", columns="variable", values="value", aggfunc="first").sort_index()

    # Ensure stable column order
    for c in TARGET_VARS:
        if c not in wide.columns:
            wide[c] = pd.NA
    wide = wide[TARGET_VARS]
    wide.index.name = "date"
    return wide

# --- run all years ---
parts = []
for y in range(START_Y, END_Y + 1):
    print(f"Fetching {y} …")
    try:
        part = fetch_daily_simple(y)
        print(f"  -> {len(part)} rows")
    except Exception as e:
        print(f"  !! {y} failed: {e}")
        part = pd.DataFrame(columns=TARGET_VARS, index=pd.Index([], name="date"))
    parts.append(part)
    time.sleep(1.0)

df = pd.concat(parts).sort_index()
df = df[~df.index.duplicated(keep="first")]

# save
df.to_csv(OUT_CSV, float_format="%.3f")
with pd.ExcelWriter(OUT_XLS, engine="openpyxl") as xlw:
    df.to_excel(xlw, sheet_name="Jokioinen_daily")

print("\nDone.")
print(f"Rows: {len(df)}, date range: {df.index.min()} → {df.index.max()}")
print(f"Saved: {OUT_CSV} and {OUT_XLS}")
df.head(12)

Fetching 1991 …
  -> 366 rows
Fetching 1992 …
  -> 367 rows
Fetching 1993 …
  -> 366 rows
Fetching 1994 …
  -> 366 rows
Fetching 1995 …
  -> 366 rows
Fetching 1996 …
  -> 367 rows
Fetching 1997 …
  -> 366 rows
Fetching 1998 …
  -> 366 rows
Fetching 1999 …
  -> 366 rows
Fetching 2000 …
  -> 367 rows
Fetching 2001 …
  -> 366 rows
Fetching 2002 …
  -> 366 rows
Fetching 2003 …
  -> 366 rows
Fetching 2004 …
  -> 367 rows
Fetching 2005 …
  -> 366 rows
Fetching 2006 …
  -> 366 rows
Fetching 2007 …
  -> 366 rows
Fetching 2008 …
  -> 367 rows
Fetching 2009 …
  -> 366 rows
Fetching 2010 …
  -> 366 rows
Fetching 2011 …
  -> 366 rows
Fetching 2012 …
  -> 367 rows
Fetching 2013 …
  -> 366 rows
Fetching 2014 …
  -> 366 rows
Fetching 2015 …
  -> 366 rows
Fetching 2016 …
  -> 367 rows
Fetching 2017 …
  -> 366 rows
Fetching 2018 …
  -> 366 rows
Fetching 2019 …
  -> 366 rows
Fetching 2020 …
  -> 367 rows
Fetching 2021 …
  -> 366 rows
Fetching 2022 …
  -> 366 rows

Done.
Rows: 11689, date range: 1991-01-

variable,rrday,snow,tday,tmin,tmax
date,,,,,
1991-01-01,0.0,27.0,-7.2,-9.3,-2.1
1991-01-02,1.6,26.0,-7.6,-13.2,-2.2
1991-01-03,7.1,28.0,-0.1,-2.9,0.6
1991-01-04,1.0,33.0,0.3,-0.1,0.6
1991-01-05,2.3,32.0,0.9,-0.1,2.0
1991-01-06,5.4,30.0,1.0,0.3,2.5
1991-01-07,0.9,32.0,0.7,0.4,1.5
1991-01-08,3.9,31.0,-0.8,-2.5,0.7
1991-01-09,10.0,35.0,0.5,0.0,0.8


In [2]:
from pathlib import Path

# 1) Choose a location outside your repo (your home directory is a safe bet)
RAW_BASE = Path.home() / "raw_data" / "climate_data"
RAW_BASE.mkdir(parents=True, exist_ok=True)

# 2) Save to this folder
csv_path = RAW_BASE / "jokioinen_daily_1991_2022.csv"
xlsx_path = RAW_BASE / "jokioinen_daily_1991_2022.xlsx"

df.to_csv(csv_path, float_format="%.3f")
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xlw:
    df.to_excel(xlw, sheet_name="Jokioinen_daily")

print(f"Saved:\n  {csv_path}\n  {xlsx_path}")

Saved:
  /home/jovyan/raw_data/climate_data/jokioinen_daily_1991_2022.csv
  /home/jovyan/raw_data/climate_data/jokioinen_daily_1991_2022.xlsx
